# Marmara University Göztepe Campus — Fiber-Optic Network Optimization
## MIS Network Analysis Notebook

This notebook walks through the Maximum Flow analysis **step by step** with
explanatory commentary between each code cell.  
For the production-grade script, see `src/solution.py`.

---

## 0 · Imports and Setup

In [ ]:
import os, csv
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DATA_FILE = os.path.join('..', 'data', 'network_data.csv')
print('Libraries loaded successfully.')

## 1 · Explore the Dataset

Before building the graph, we inspect the CSV to understand what each column represents.

| Column | Description |
|---|---|
| `source` / `target` | Campus building IDs (nodes) |
| `capacity_mbps` | Maximum fiber throughput — the **key constraint** for max-flow |
| `distance_m` | Physical cable run length (metadata only) |
| `cable_cost_tl` | Estimated installation cost in Turkish Lira (metadata only) |
| `node_type_*` | Role of each building (core / admin / faculty / lab / service) |

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f'Shape: {df.shape[0]} edges × {df.shape[1]} columns')
df.head(14)

In [ ]:
# Summary statistics — understand the capacity distribution
print('Capacity distribution (Mbps):')
print(df['capacity_mbps'].describe().astype(int).to_string())
print(f'\nTotal infrastructure cost: ₺{df["cable_cost_tl"].sum():,}')

## 2 · Build the Directed Graph

Each CSV row becomes **two directed arcs** (forward + reverse) to model
the bidirectional nature of fiber-optic cable.

The `capacity` attribute on each arc is what the Maximum Flow solver will
respect as an upper bound.

In [ ]:
G = nx.DiGraph()

with open(DATA_FILE, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        u, v = row['source'].strip(), row['target'].strip()
        cap  = int(row['capacity_mbps'])
        G.add_node(u, node_type=row['node_type_source'].strip())
        G.add_node(v, node_type=row['node_type_target'].strip())
        G.add_edge(u, v, capacity=cap,
                   distance_m=int(row['distance_m']),
                   cable_cost_tl=int(row['cable_cost_tl']))
        if not G.has_edge(v, u):
            G.add_edge(v, u, capacity=cap,
                       distance_m=int(row['distance_m']),
                       cable_cost_tl=int(row['cable_cost_tl']))

print(f'Nodes : {G.number_of_nodes()}')
print(f'Edges : {G.number_of_edges()} directed arcs')
print(f'Nodes : {list(G.nodes())}')

## 3 · Add the Super-Sink

NetworkX's max-flow functions require **exactly one source and one sink**.
We have two sink nodes (`student_labs_A`, `student_labs_B`), so we create
a virtual `super_sink` node and connect both labs to it with infinite capacity.

This lets the algorithm treat both lab blocks as a single aggregate demand point
without artificially constraining the merge link.

In [ ]:
SOURCE     = 'data_center'
SINK_NODES = ['student_labs_A', 'student_labs_B']
SUPER_SINK = 'super_sink'

G.add_node(SUPER_SINK, node_type='virtual')
for lab in SINK_NODES:
    G.add_edge(lab, SUPER_SINK, capacity=float('inf'))

print(f'Super-sink added.  Total nodes now: {G.number_of_nodes()}')

## 4 · Solve Maximum Flow (Edmonds-Karp)

We call `nx.maximum_flow()` with the Edmonds-Karp flow function.

**Returns:**
- `flow_value` — total Mbps deliverable from source to sink
- `flow_dict`  — dictionary `{u: {v: flow_on_uv}}` for every arc

In [ ]:
flow_value, flow_dict = nx.maximum_flow(
    G, SOURCE, SUPER_SINK,
    flow_func=nx.algorithms.flow.edmonds_karp
)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  MAXIMUM FLOW  :  {int(flow_value):,} Mbps  ({flow_value/1000:.1f} Gbps)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

## 5 · Inspect Per-Edge Flow Allocation

In [ ]:
rows = []
for u in sorted(flow_dict):
    for v, flow in sorted(flow_dict[u].items()):
        if 'super_sink' in (u, v):
            continue
        cap = G[u][v].get('capacity', float('inf'))
        if cap == float('inf'):
            continue
        rows.append({
            'From': u, 'To': v,
            'Flow (Mbps)': int(flow),
            'Capacity (Mbps)': int(cap),
            'Utilisation (%)': round(flow / cap * 100, 1)
        })

result_df = pd.DataFrame(rows)
result_df = result_df.sort_values('Utilisation (%)', ascending=False)
result_df.style.bar(subset=['Utilisation (%)'], color='#E74C3C', vmin=0, vmax=100)

## 6 · Minimum Cut Analysis

The **Min-Cut** tells us which edges are bottlenecks.  
By the **Max-Flow Min-Cut Theorem**, the min-cut value equals the max-flow value.

In [ ]:
cut_value, (reachable, non_reachable) = nx.minimum_cut(G, SOURCE, SUPER_SINK)

print(f'Min-cut value : {int(cut_value):,} Mbps  (= max flow ✓)')
print(f'\nSource side   : {sorted(n for n in reachable if n != SUPER_SINK)}')
print(f'Sink side     : {sorted(n for n in non_reachable if n != SUPER_SINK)}')

print('\nBottleneck edges crossing the cut:')
for u in reachable:
    for v in G.successors(u):
        if v in non_reachable:
            cap = G[u][v].get('capacity', float('inf'))
            if cap < float('inf'):
                print(f'  ✦ {u}  →  {v}   [{cap:,} Mbps]')

## 7 · Visualisation

In [ ]:
# Run the full visualization from solution.py
import sys
sys.path.insert(0, os.path.join('..', 'src'))
import importlib.util, types

spec = importlib.util.spec_from_file_location(
    'solution',
    os.path.join('..', 'src', 'solution.py')
)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)

fig_path = os.path.join('..', 'results', 'network_visualization.png')
sol.visualise(G, flow_dict, reachable, SOURCE, SUPER_SINK, fig_path)

from IPython.display import Image
Image(fig_path)

## 8 · Key Insights

| Insight | Value |
|---|---|
| Maximum achievable throughput | **13,000 Mbps** |
| Primary bottleneck | `data_center → engineering` (10 Gbps, 100% saturated) |
| Secondary bottleneck | `science_letters → student_labs_A` (2.5 Gbps, 100% saturated) |
| Idle capacity | `rectorate`, `library`, `student_affairs` corridors carry **0 flow** |
| Upgrade recommendation | Add a second 10 GbE link: `data_center → student_labs_A/B` directly |

> **Managerial takeaway:** The network has a single dominant path carrying 77% of all student-lab traffic. Redundancy and capacity upgrades on the `data_center ↔ engineering` segment should be the top capital expenditure priority for the IT directorate.